In [3]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ============================================================
# LOAD
# ============================================================
truth_full = pd.read_parquet(r"raw/truth_match/truth_tract3828.parquet")
print(f"Loaded truth_full: {len(truth_full):,} rows")

cleaned = pd.read_parquet(r"processed/cleaned_catalog.parquet")
print(f"Loaded cleaned: {len(cleaned):,} rows")
print(f"cleaned already has native 'blendedness' (SCARLET/pipeline-measured): {'blendedness' in cleaned.columns}")


# ============================================================
# BLENDEDNESS FUNCTION — Duan et al. (2026), Section 4.8
# blendedness = 1 - I_child / I_parent
# computed on the FULL uncut truth catalog so faint neighbors count
# ============================================================
def compute_blendedness_fast(df, ra_col="ra", dec_col="dec",
                              flux_col="flux_r", sigma_px=6.0,
                              trunc_px=30.0, pixel_scale=0.2):
    ra = df[ra_col].values
    dec = df[dec_col].values
    flux = df[flux_col].values
    n = len(df)
    dec_mean = np.deg2rad(dec.mean())
    x = (ra - ra.mean()) * np.cos(dec_mean) * 3600.0 / pixel_scale
    y = (dec - dec.mean()) * 3600.0 / pixel_scale
    coords = np.column_stack([x, y])
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=trunc_px, output_type="ndarray")
    print(f"Found {len(pairs):,} neighbor pairs among {n:,} sources")
    i_idx = pairs[:, 0]
    j_idx = pairs[:, 1]
    d = np.linalg.norm(coords[i_idx] - coords[j_idx], axis=1)
    w = np.exp(-0.5 * (d / sigma_px) ** 2)
    neighbor_flux_sum = np.zeros(n)
    np.add.at(neighbor_flux_sum, i_idx, w * flux[j_idx])
    np.add.at(neighbor_flux_sum, j_idx, w * flux[i_idx])
    i_parent = flux + neighbor_flux_sum
    with np.errstate(divide='ignore', invalid='ignore'):
        blendedness = np.where(i_parent > 0, 1.0 - (flux / i_parent), 0.0)
    df["blendedness"] = blendedness
    return df


# ============================================================
# STEP 1: compute truth-based blendedness on FULL truth catalog
# (raw truth_tract3828.parquet uses plain 'ra'/'dec', NOT 'ra_true'/'dec_true' —
#  those renamed versions only exist after the cross-match into cleaned_catalog)
# ============================================================
truth_full = compute_blendedness_fast(truth_full, ra_col="ra", dec_col="dec", flux_col="flux_r")


# ============================================================
# STEP 2: merge onto cleaned catalog as blendedness_truth
# (cleaned already has its own native 'blendedness' from SCARLET/the pipeline —
#  keep both, rename the merged-in one so nothing collides)
# ============================================================
blend_lookup = truth_full[["id", "blendedness"]].drop_duplicates(subset="id")
blend_lookup = blend_lookup.rename(columns={"blendedness": "blendedness_truth"})

cleaned = cleaned.merge(blend_lookup, on="id", how="left")

n_missing = cleaned["blendedness_truth"].isna().sum()
print(f"\nRows with no truth-based blendedness match: {n_missing:,} / {len(cleaned):,}")

cleaned_matched = cleaned.dropna(subset=["blendedness_truth"]).copy()
print(f"Proceeding with {len(cleaned_matched):,} matched rows")

# derive mag_r for verification (from truth flux) — guarded against zero/negative flux
cleaned_matched["mag_r"] = np.where(
    cleaned_matched["flux_r"] > 0,
    -2.5 * np.log10(cleaned_matched["flux_r"]) + 31.4,
    np.nan
)
n_bad_flux = cleaned_matched["mag_r"].isna().sum()
if n_bad_flux > 0:
    print(f"Note: {n_bad_flux} rows had zero/negative flux_r -> mag_r set to NaN")


# ============================================================
# STEP 3: agreement check — truth-based vs. SCARLET/pipeline-native blendedness
# (diagnostic only — NOT used for patch selection, to avoid circularity
#  since SCARLET is one of the models under evaluation)
# ============================================================
print("\n" + "="*60)
print("AGREEMENT CHECK: blendedness_truth vs. native (SCARLET) blendedness")
print("="*60)

both = cleaned_matched.dropna(subset=["blendedness", "blendedness_truth"])
corr = both["blendedness"].corr(both["blendedness_truth"])
print(f"Pearson correlation: {corr:.4f}")

mean_abs_diff = (both["blendedness"] - both["blendedness_truth"]).abs().mean()
print(f"Mean absolute difference: {mean_abs_diff:.4f}")

native_unblended = both["blendedness"] < 0.02
truth_unblended = both["blendedness_truth"] < 0.02
agree_frac = (native_unblended == truth_unblended).mean()
print(f"Classification agreement (unblended/blended @ 0.02 cut): {agree_frac:.4f}")

print("\nSide-by-side distributions:")
print(both[["blendedness", "blendedness_truth"]].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))


# ============================================================
# STEP 4: patch density + tier selection using blendedness_truth
# (ground-truth based -> avoids circularity with SCARLET being under evaluation)
# ============================================================
BLEND_THRESHOLD = 0.02  # Duan et al. (2026) unblended/blended boundary

def compute_patch_density(df, patch_col="patch_true", blend_col="blendedness_truth"):
    density = (
        df.groupby(patch_col)[blend_col]
        .apply(lambda x: (x > BLEND_THRESHOLD).mean())
        .sort_values(ascending=False)
    )
    return density

def select_high_blend_patches(patch_density, n=5):
    selected = patch_density.head(n)
    return selected.index.tolist(), selected

def select_mean_blend_patches(patch_density, n=5):
    mean_density = patch_density.mean()
    distance_from_mean = (patch_density - mean_density).abs().sort_values()
    selected_ids = distance_from_mean.head(n).index.tolist()
    selected = patch_density.loc[selected_ids].sort_values(ascending=False)
    return selected_ids, selected, mean_density

def select_low_blend_patches(patch_density, n=5):
    selected = patch_density.tail(n)
    return selected.index.tolist(), selected

patch_density = compute_patch_density(cleaned_matched)

high_patches, high_table = select_high_blend_patches(patch_density)
print("\nHIGH — 5 most-blended patches (blendedness_truth):")
print(high_table)
print("Selected:", high_patches)

mean_patches, mean_table, mean_value = select_mean_blend_patches(patch_density)
print(f"\nMean blending density across all patches: {mean_value:.4f}")
print("MEAN — 5 patches closest to average blending density:")
print(mean_table)
print("Selected:", mean_patches)

low_patches, low_table = select_low_blend_patches(patch_density)
print("\nLOW — 5 least-blended patches:")
print(low_table)
print("Selected:", low_patches)


# ============================================================
# STEP 5: verification against Duan et al. (2026)
# ============================================================
print("\n" + "="*60)
print("VERIFICATION AGAINST DUAN ET AL. (2026)")
print("="*60)

frac_bright    = (cleaned_matched["mag_r"] < 24.5).mean()
frac_unblended = (cleaned_matched["blendedness_truth"] < 0.3).mean()
frac_strict    = (cleaned_matched["blendedness_truth"] < 0.02).mean()

print(f"frac mag_r < 24.5              (paper: 21.4%, expect ~100% since catalog pre-cut): {frac_bright:.4f}")
print(f"frac blendedness_truth < 0.3   (paper: 77.5%): {frac_unblended:.4f}")
print(f"frac blendedness_truth < 0.02  (unblended, strict): {frac_strict:.4f}")

assert cleaned_matched["blendedness_truth"].min() >= 0.0, "blendedness < 0 — check flux sign"
assert cleaned_matched["blendedness_truth"].max() <= 1.0, "blendedness > 1 — check i_parent computation"

print("\nDistribution shape (blendedness_truth):")
print(cleaned_matched["blendedness_truth"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))

spread = patch_density.max() - patch_density.min()
print(f"\nPatch density spread (max-min): {spread:.4f}")
if spread < 0.05:
    print("WARNING: patches barely differ from each other — check for saturation/inflation.")
else:
    print("OK: meaningful spread across patches — tier separation looks real.")


# ============================================================
# STEP 6: save final patch selection + cleaned_matched w/ blendedness_truth
# ============================================================
final_patches = {
    "high": high_patches,
    "mean": mean_patches,
    "low": low_patches,
}
print("\nFinal 15-patch selection:", final_patches)

cleaned_matched.to_parquet(r"processed/cleaned_catalog_with_blendedness_truth.parquet", index=False)
print("Saved: processed/cleaned_catalog_with_blendedness_truth.parquet")

Loaded truth_full: 4,645,754 rows
Loaded cleaned: 132,830 rows
cleaned already has native 'blendedness' (SCARLET/pipeline-measured): True
Found 43,370,372 neighbor pairs among 4,645,754 sources

Rows with no truth-based blendedness match: 0 / 132,830
Proceeding with 132,830 matched rows

AGREEMENT CHECK: blendedness_truth vs. native (SCARLET) blendedness
Pearson correlation: 0.5129
Mean absolute difference: 0.0758
Classification agreement (unblended/blended @ 0.02 cut): 0.4499

Side-by-side distributions:
         blendedness  blendedness_truth
count  132827.000000      132827.000000
mean        0.026554           0.089270
std         0.072930           0.122089
min        -0.010123           0.000001
10%         0.000000           0.005383
25%         0.000004           0.016102
50%         0.000503           0.044174
75%         0.010272           0.108338
90%         0.080800           0.230225
95%         0.166438           0.344623
99%         0.376864           0.592928
max      